# Reset index of alignment

Rearr does not support mutiple references for single query read. As a workaround, we repeat a query read multiple times if it has multiple corresponding references. This step assign the same index for the same query read. For a query read with multiple references, we distribute the count of the query read to each reference as follows:
  - normalize the reference count to get the priori distribution for query count;
  - normalize the alignment score by a temperature and use softmax to calculate the conditional probability;
  - mix the priori distribution and conditional probability to get the posteriori distribution.

In [ ]:
from importlib import resources

import yaml

from naapam import analyze

with resources.as_file(
    resources.files("naapam.filter_configs") / "analyze.correct_alg.yaml"
) as pf:
    with pf.open("r") as fd:
        params = yaml.load(fd, Loader=yaml.CLoader)

analyze.correct_alg(root_dir="/home/ljw/sdb1/naapam", **params)

# Summarize the distribution of alignment score

We summarize the distribution of alignment score to determine a suitable threshold for the alignment score.

In [ ]:
from naapam import analyze

analyze.stat_read(root_dir="/home/ljw/sdb1/naapam")

# Collect alignment results

We collect the corrected alignment and apply a read-wise filter based on the alignment score. We then aggregate all reads based on the mutant type near the cleavage site predicted by rearr.

In [ ]:
from importlib import resources

import yaml

from naapam import analyze

with resources.as_file(
    resources.files("naapam.filter_configs") / "analyze.collect_data.yaml"
) as pf:
    with pf.open("r") as fd:
        params = yaml.load(fd, Loader=yaml.CLoader)

analyze.collect_data(root_dir="/home/ljw/sdb1/naapam", **params)

# Summarize reference properties

We summarize the properties per reference per sample. The properties includes:
  - the total read count;
  - the mutant frequency;
  - the frequency of the upstream deletion size per downstream templated insertion size.

In [ ]:
from importlib import resources

import yaml

from naapam import analyze

with resources.as_file(
    resources.files("naapam.filter_configs") / "analyze.stat_ref.yaml"
) as pf:
    with pf.open("r") as fd:
        params = yaml.load(fd, Loader=yaml.CLoader)
        min_count_tot = params["min_count_tot"]

analyze.stat_ref(root_dir="/home/ljw/sdb1/naapam", **params)

# Fit a model to predict relative abundance from upstream deletion size

We count reads for each upstream deletion size and calculate their relative abundance to the upstream blunt end counts. We find that the relative abundance decreases exponentially with the upstream deletion size. Thus, we first a linear model between the log of the relative abundance and the upstream deletion size.

In [ ]:
from importlib import resources

import yaml

from naapam import analyze

with resources.as_file(
    resources.files("naapam.filter_configs") / "analyze.stat_ref.yaml"
) as pf:
    with pf.open("r") as fd:
        params = yaml.load(fd, Loader=yaml.CLoader)
        max_up_del_size = params["max_up_del_size"]

analyze.fit_ref(root_dir="/home/ljw/sdb1/naapam", max_up_del_size=max_up_del_size)

# Filter out references

We first filter out references with low total read count or low wild-type frequencies. Then we use the upper bound of the 95% prediction interval to calculate the upper limit of the relative abundance to the upstream blunt end counts for each upstream deletion size. The references with abnormally high abundance at any upstream deletion size are filtered out.

In [ ]:
from importlib import resources

import yaml

from naapam import analyze

with resources.as_file(
    resources.files("naapam.filter_configs") / "analyze.stat_ref.yaml"
) as pf:
    with pf.open("r") as fd:
        params = yaml.load(fd, Loader=yaml.CLoader)
        min_count_tot = params["min_count_tot"]

with resources.as_file(
    resources.files("naapam.filter_configs") / "analyze.filter_ref.yaml"
) as pf:
    with pf.open("r") as fd:
        params = yaml.load(fd, Loader=yaml.CLoader)

analyze.filter_ref(
    root_dir="/home/ljw/sdb1/naapam", **params, min_count_tot=min_count_tot
)

# Summarize properties of mutants

We summarize properties of mutants to determine proper filtering thresholds. The properties includes:
  - the up/down-stream deletion size;
  - the random insertion size;
  - the ratio among all mutants of the reference.

In [ ]:
from importlib import resources

import yaml

from naapam import analyze

with resources.as_file(
    resources.files("naapam.filter_configs") / "analyze.stat_ref.yaml"
) as pf:
    with pf.open("r") as fd:
        params = yaml.load(fd, Loader=yaml.CLoader)

analyze.stat_mutant(
    root_dir="/home/ljw/sdb1/naapam", min_count_tot=params["min_count_tot"]
)

# Filter mutant

We filter out mutants if:
  - the up/down-stream deletion size is too larege;
  - the random insertion size is too large;
  - the ratio among all mutants of the reference is too high.

In [ ]:
from importlib import resources

import yaml

from naapam import analyze

with resources.as_file(
    resources.files("naapam.filter_configs") / "analyze.stat_ref.yaml"
) as pf:
    with pf.open("r") as fd:
        params = yaml.load(fd, Loader=yaml.CLoader)
        max_up_del_size = params["max_up_del_size"]

with resources.as_file(
    resources.files("naapam.filter_configs") / "analyze.filter_mutant.yaml"
) as pf:
    with pf.open("r") as fd:
        params = yaml.load(fd, Loader=yaml.CLoader)

analyze.filter_mutant(
    root_dir="/home/ljw/sdb1/naapam", max_up_del_size=max_up_del_size, **params
)

# Duplicate treat

We duplicate each treat sample to wt1 and wt2. Some treat samples are further duplicated to wt11 and wt21.

In [ ]:
from naapam import mix

mix.duplicate_treat(root_dir="/home/ljw/sdb1/naapam")

# Duplicate control

Some control does not have all the four time points. We duplicate control samples from exist time potins to missing ones.

In [ ]:
from naapam import mix

mix.duplicate_control(root_dir="/home/ljw/sdb1/naapam")

# Merge duplicated treat and control

We then merge duplcated treat and control to apply the normalization of Kim.

In [ ]:
from naapam import mix

mix.merge(root_dir="/home/ljw/sdb1/naapam")

# Kim correction

We apply the Kim's correction method to our samples. For each mutant type, we calculate the expected count in the treat sample according to its frequency in the control sample. We then substract the expected count from the observed count in the treat sample. The negative results is corrected to zero.

In [ ]:
from naapam import analyze

analyze.kim_correct(root_dir="/home/ljw/sdb1/naapam")

# Draw mean abundance over upstream deletion size

For each downstream templated insertion size, we draw the mean abundance for each upstream deletion size to see whether there are artifacts due to the error during chip synthesis.

In [ ]:
from naapam import draw

for target in ["count", "freq_kim"]:
    draw.mean_over_up_del_size_on_tem(
        root_dir="/home/ljw/sdb1/naapam", target=target, stem_wise=False
    )

# Annote necessary columns

To support downstream analysis, we annote additional columns like barcode id, barcode sequence, spacer sequence, reference to the final results.

In [ ]:
from naapam import analyze

analyze.annote_columns(root_dir="/home/ljw/sdb1/naapam")